# Kindle Data Investigation

Explore and analyze Kindle reading data from the Amazon data export.

In [1]:
import fs from 'node:fs';
import decompress from "npm:decompress";

type FileResult =
  | { success: true; filePath: string }
  | { success: false; error: 'NOT_FOUND' | 'PERMISSION_DENIED' };

function getKindleZip(): FileResult {
  const directory = './data/kindle';
  const files = fs.readdirSync(directory);
  const matches = files.filter(file => file.endsWith(".zip"));
  if (matches.length == 0) {
    return { success: false, error: "NOT_FOUND" }
  }
  return { success: true, filePath: './data/kindle/' + matches[0] }
}

function clearDirIfExists(dir: string) {
  if (fs.existsSync(dir)) {
    console.log("Cleared contents of ", dir)
    fs.rmSync(dir, { recursive: true, force: true });
  }
}

const kindleZip: FileResult = getKindleZip()
const targetDir = "./data/kindle/output"
if (kindleZip.success) {
  console.log("Unzipping file : ", kindleZip.filePath)
  clearDirIfExists(targetDir)
  await decompress(kindleZip.filePath, targetDir);
  console.log("Done!")
} else {
  console.warn("Couldn't find kindle zip")
}

Unzipping file :  ./data/kindle/Kindle (1).zip
Cleared contents of  ./data/kindle/output
Done!


In [2]:
import pl from 'npm:nodejs-polars';
import { displayTable } from "./utils.ts";

// Reading sessions with book titles - this is the juiciest dataset
const sessionsDf = pl.readCSV("./data/kindle/output/Kindle.ReadingInsights/datasets/Kindle.reading-insights-sessions_with_adjustments/Kindle.reading-insights-sessions_with_adjustments.csv")

console.log("Columns:", sessionsDf.columns)
displayTable(sessionsDf)

Columns: [
  "ASIN",
  "end_time",
  "product_name",
  "reading_marketplace",
  "start_time",
  "total_reading_milliseconds"
]
{ height: 3131, width: 6 }
┌───────┬──────────────┬────────────────────────────┬─────────────────────────────────────┬─────────────────────┬────────────────────────────┬────────────────────────────┐
│ (idx) │ ASIN         │ end_time                   │ product_name                        │ reading_marketplace │ start_time                 │ total_reading_milliseconds │
├───────┼──────────────┼────────────────────────────┼─────────────────────────────────────┼─────────────────────┼────────────────────────────┼────────────────────────────┤
│     0 │ "B0BVRV7W7C" │ "2026-01-08T08:40:22.900Z" │ "Antimatter Blues: A Mickey7 Novel" │ "www.amazon.co.uk"  │ "2026-01-08T08:40:19.200Z" │                       3700 │
│     1 │ "B0BVRV7W7C" │ "2026-01-08T08:38:05.200Z" │ "Antimatter Blues: A Mickey7 Novel" │ "www.amazon.co.uk"  │ "2026-01-08T08:36:08.100Z" │                

In [4]:
// Total reading time per book
const readingByBook = sessionsDf
  .groupBy("product_name")
  .agg(
    pl.col("total_reading_milliseconds").sum().alias("total_ms"),
    pl.col("ASIN").count().alias("session_count")
  )
  .withColumn(
    pl.col("total_ms").div(3600000).alias("total_hours")
  )
  .sort("total_ms", true)

console.log("Reading time per book (hours):")
displayTable(readingByBook, 100)

Reading time per book (hours):
{ height: 105, width: 4 }
┌───────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┬───────────────┬──────────────────────┐
│ (idx) │ product_name                                                                                                                                       │ total_ms  │ session_count │ total_hours          │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────┼───────────────┼──────────────────────┤
│     0 │ "Not Available"                                                                                                                                    │ 456551200 │           440 │ 126.81977777777777   │
│     1 │ "The Eye of the World"                                                                       

In [5]:
// Total reading stats
const totalStats = sessionsDf.select(
  pl.col("total_reading_milliseconds").sum().div(3600000).alias("total_hours_reading"),
  pl.col("total_reading_milliseconds").count().alias("total_sessions"),
  pl.col("product_name").nUnique().alias("unique_books")
)
console.log("Overall reading stats:")
displayTable(totalStats)

Overall reading stats:
{ height: 1, width: 3 }
┌───────┬─────────────────────┬────────────────┬──────────────┐
│ (idx) │ total_hours_reading │ total_sessions │ unique_books │
├───────┼─────────────────────┼────────────────┼──────────────┤
│     0 │ 787.2131111111111   │           3131 │          105 │
└───────┴─────────────────────┴────────────────┴──────────────┘


In [6]:
// Reading sessions over time - parse start_time and group by date
const sessionsWithDate = sessionsDf
  .withColumn(
    pl.col("start_time").str.slice(0, 10).alias("date")
  )

const dailyReading = sessionsWithDate
  .groupBy("date")
  .agg(
    pl.col("total_reading_milliseconds").sum().div(60000).alias("minutes_read"),
    pl.col("ASIN").count().alias("sessions")
  )
  .sort("date")

console.log("Daily reading (minutes):")
displayTable(dailyReading, 20)

Daily reading (minutes):
{ height: 837, width: 3 }
┌───────┬──────────────┬────────────────────┬──────────┐
│ (idx) │ date         │ minutes_read       │ sessions │
├───────┼──────────────┼────────────────────┼──────────┤
│     0 │ "2020-12-27" │ 151.88333333333335 │       26 │
│     1 │ "2020-12-28" │ 139.79333333333335 │       14 │
│     2 │ "2020-12-29" │ 154.645            │       15 │
│     3 │ "2020-12-30" │ 142.54833333333335 │        5 │
│     4 │ "2020-12-31" │ 83.745             │        4 │
│     5 │ "2021-01-01" │ 41.78333333333334  │        1 │
│     6 │ "2021-01-02" │ 183.51166666666668 │       15 │
│     7 │ "2021-01-03" │ 159.475            │        4 │
│     8 │ "2021-01-04" │ 400.35             │       15 │
│     9 │ "2021-01-05" │ 240.97833333333335 │       12 │
│    10 │ "2021-01-06" │ 350.83000000000004 │       12 │
│    11 │ "2021-01-07" │ 276.96000000000004 │       10 │
│    12 │ "2021-01-08" │ 83.2               │        6 │
│    13 │ "2021-01-09" │ 139.405     

In [7]:
// Monthly reading breakdown
const sessionsWithMonth = sessionsDf
  .withColumn(
    pl.col("start_time").str.slice(0, 7).alias("month")
  )

const monthlyReading = sessionsWithMonth
  .groupBy("month")
  .agg(
    pl.col("total_reading_milliseconds").sum().div(3600000).alias("hours_read"),
    pl.col("product_name").nUnique().alias("unique_books"),
    pl.col("ASIN").count().alias("sessions")
  )
  .sort("month")

console.log("Monthly reading:")
displayTable(monthlyReading, 24)

Monthly reading:
{ height: 52, width: 4 }
┌───────┬───────────┬───────────────────────┬──────────────┬──────────┐
│ (idx) │ month     │ hours_read            │ unique_books │ sessions │
├───────┼───────────┼───────────────────────┼──────────────┼──────────┤
│     0 │ "2020-12" │ 11.210249999999998    │            1 │       64 │
│     1 │ "2021-01" │ 65.48949999999999     │            5 │      155 │
│     2 │ "2021-02" │ 10.975444444444443    │            2 │       45 │
│     3 │ "2021-03" │ 28.994833333333332    │            3 │      118 │
│     4 │ "2021-04" │ 24.28983333333333     │            5 │       82 │
│     5 │ "2021-05" │ 26.53088888888889     │            9 │       83 │
│     6 │ "2021-06" │ 0.020138888888888887  │            1 │        2 │
│     7 │ "2021-07" │ 1.9139722222222222    │            5 │       11 │
│     8 │ "2021-08" │ 0.004555555555555555  │            1 │        1 │
│     9 │ "2021-11" │ 6.102055555555555     │            1 │       20 │
│    10 │ "2021-12" │ 

In [8]:
// Books completed
const completedDf = pl.readCSV("./data/kindle/output/Kindle.ReadingInsights/datasets/Kindle.UserUniqueTitlesCompleted/Kindle.UserUniqueTitlesCompleted.csv")

console.log("Books completed:", completedDf.shape)
displayTable(completedDf, 60)

Books completed: { height: 58, width: 2 }
{ height: 58, width: 2 }
┌───────┬───────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ (idx) │ asin_date_and_content_type        │ product_name                                                                                                                                       │
├───────┼───────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│     0 │ "B002RI9KAE_2025-04-23_AUTOMATIC" │ "Snow Crash"                                                                                                                                       │
│     1 │ "B002VBV1HC_2024-10-23_AUTOMATIC" │ "The Eye of the World"                                                                                     

In [9]:
// Days where reading was tracked
const dayUnitsDf = pl.readCSV("./data/kindle/output/Kindle.ReadingInsights/datasets/Kindle.ReadingInsightsDayUnits/Kindle.ReadingInsightsDayUnits.csv")

console.log("Total days with reading tracked:", dayUnitsDf.shape.height)
displayTable(dayUnitsDf, 10)

Total days with reading tracked: 719
{ height: 719, width: 1 }
┌───────┬────────────────────────┐
│ (idx) │ reading_tracked_on_day │
├───────┼────────────────────────┤
│     0 │ "2026-01-08T00:00:00Z" │
│     1 │ "2026-01-07T00:00:00Z" │
│     2 │ "2026-01-03T00:00:00Z" │
│     3 │ "2026-01-02T00:00:00Z" │
│     4 │ "2025-12-31T00:00:00Z" │
│     5 │ "2025-12-30T00:00:00Z" │
│     6 │ "2025-12-29T00:00:00Z" │
│     7 │ "2025-12-28T00:00:00Z" │
│     8 │ "2025-12-27T00:00:00Z" │
│     9 │ "2025-12-25T00:00:00Z" │
└───────┴────────────────────────┘


In [10]:
// Highlight actions
const highlightsDf = pl.readCSV("./data/kindle/output/Kindle.Devices.kindleHighlightActions/Kindle.Devices.kindleHighlightActions.csv")

console.log("Highlight actions:", highlightsDf.shape)
console.log("Columns:", highlightsDf.columns)
displayTable(highlightsDf, 10)

Highlight actions: { height: 84, width: 11 }
Columns: [
  "created_timestamp",
  "ASIN",
  "customer_context",
  "action",
  "highlight_color",
  "is_starred",
  "show_options_after_highlighting",
  "number_of_words_in_highlight",
  "device_family",
  "country",
  "preferred_marketplace"
]
{ height: 84, width: 11 }
┌───────┬────────────────────────┬──────────────┬─────────────────────────────────────┬────────────────────┬─────────────────┬────────────┬────────────────────────────────────────────────────────────┬──────────────────────────────┬──────────────────────┬─────────┬───────────────────────┐
│ (idx) │ created_timestamp      │ ASIN         │ customer_context                    │ action             │ highlight_color │ is_starred │ show_options_after_highlighting                            │ number_of_words_in_highlight │ device_family        │ country │ preferred_marketplace │
├───────┼────────────────────────┼──────────────┼─────────────────────────────────────┼──────────────────

In [11]:
// Kindle achievements / book rewards
const achievementsDf = pl.readCSV("./data/kindle/output/Kindle.BookRewards/datasets/Kindle.BookRewards.Achievements.2/Kindle.BookRewards.Achievements.2.csv")

console.log("Kindle achievements:")
displayTable(achievementsDf, 25)

Kindle achievements:
{ height: 21, width: 5 }
┌───────┬──────────────────────────────────┬─────────────────┬────────────────────────┬────────────────────┬──────────┐
│ (idx) │ AchievementGroupName             │ AchievementName │ EarnDate               │ Marketplace        │ Quantity │
├───────┼──────────────────────────────────┼─────────────────┼────────────────────────┼────────────────────┼──────────┤
│     0 │ "2024 Year End Kindle Challenge" │ "GoldReader"    │ "2024-12-17T23:04:05Z" │ "www.amazon.co.uk" │        1 │
│     1 │ "2024 Year End Kindle Challenge" │ "PerfectMonth"  │ "2024-11-30T01:17:55Z" │ "www.amazon.co.uk" │        1 │
│     2 │ "2024 Year End Kindle Challenge" │ "SilverReader"  │ "2024-11-12T21:36:47Z" │ "www.amazon.co.uk" │        1 │
│     3 │ "2024 Year End Kindle Challenge" │ "Bibliophile"   │ "2024-11-10T12:43:20Z" │ "www.amazon.co.uk" │        1 │
│     4 │ "2024 Year End Kindle Challenge" │ "Bookworm"      │ "2024-11-02T20:45:40Z" │ "www.amazon.co.uk" │      

In [12]:
// E-reader device active usage time
const usageTimeDf = pl.readCSV("./data/kindle/output/Kindle.Devices.EreaderDeviceActiveUsageTime/Kindle.Devices.EreaderDeviceActiveUsageTime.csv")

const usageWithDate = usageTimeDf
  .withColumn(
    pl.col("created_timestamp").str.slice(0, 7).alias("month")
  )
  .withColumn(
    pl.col("device_active_time_in_ms").div(3600000).alias("hours")
  )

const monthlyUsage = usageWithDate
  .groupBy("month")
  .agg(
    pl.col("hours").sum().alias("total_hours"),
    pl.col("device_active_time_in_ms").count().alias("sessions")
  )
  .sort("month")

console.log("Monthly e-reader device usage:")
displayTable(monthlyUsage, 24)

Monthly e-reader device usage:
{ height: 12, width: 3 }
┌───────┬───────────┬────────────────────┬──────────┐
│ (idx) │ month     │ total_hours        │ sessions │
├───────┼───────────┼────────────────────┼──────────┤
│     0 │ "2023-09" │ 5.655476111111111  │       24 │
│     1 │ "2023-10" │ 22.656052499999998 │       66 │
│     2 │ "2023-11" │ 21.71990861111111  │       43 │
│     3 │ "2023-12" │ 12.126626388888887 │       35 │
│     4 │ "2024-01" │ 44.380293888888886 │      127 │
│     5 │ "2024-02" │ 7.442349444444444  │       29 │
│     6 │ "2024-04" │ 4.760295277777777  │       12 │
│     7 │ "2024-05" │ 19.8169175         │       77 │
│     8 │ "2024-06" │ 2.926769722222222  │       20 │
│     9 │ "2024-07" │ 1.2228083333333333 │        3 │
│    10 │ "2024-08" │ 1.8503130555555554 │        8 │
│    11 │ "2024-09" │ 0.3959008333333333 │        2 │
└───────┴───────────┴────────────────────┴──────────┘


In [13]:
// Whispersync data - books and annotations
const whispersyncDf = pl.readCSV("./data/kindle/output/Digital.Content.Whispersync/whispersync.csv")

console.log("Whispersync records:", whispersyncDf.shape)
console.log("Columns:", whispersyncDf.columns)

// Unique books in whispersync
const whispersyncBooks = whispersyncDf
  .select("Product Name", "Annotation Type")
  .unique()
  .sort("Product Name")

console.log("Unique books with annotations:")
displayTable(whispersyncBooks, 30)

Whispersync records: { height: 497, width: 16 }
Columns: [
  "ASIN",
  "Non ASIN",
  "Product Name",
  "Annotation Type",
  "Customer modified date on device",
  "Device Serial Number",
  "Third Party Device",
  "ContentType",
  "Format",
  "Creation Date",
  "LastUpdatedDate",
  "Highlight",
  "Version",
  "Selection Type",
  "Note",
  "Is Deleted"
]
Unique books with annotations:
{ height: 347, width: 2 }
┌───────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────┐
│ (idx) │ Product Name                                                                                                                            │ Annotation Type           │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────────────────┤
│     0 │ "11.22.63"                                         

In [14]:
// Reading behavior summary stats
const behaviorDf = pl.readCSV("./data/kindle/output/Kindle.ReadingBehaviorCounts/Kindle.ReadingBehaviorCounts.csv")

// This has genre breakdowns - let's look at those
const genreCols = behaviorDf.columns.filter(c => c.includes("Genre") && c.includes("Count"))
console.log("Genre count columns:", genreCols)

// Get the latest snapshot
const latestBehavior = behaviorDf.sort("Snapshot Date (in UTC)", true).head(1)

// Extract genre counts into something displayable
const genreData = genreCols.map(col => {
  const count = latestBehavior.getColumn(col).get(0)
  const genre = col.replace("Books Read Count (", "").replace(" Genre)", "").replace(")", "")
  return { genre, count }
}).filter(g => Number(g.count) > 0)

console.log("Genre breakdown (latest snapshot):")
console.table(genreData)

Genre count columns: [
  "Books Read Count (Fiction Genre)",
  "Books Read Count (Children's Genre)",
  "Books Read Count (Nonfiction Genre)",
  "Books Read Count (Comics Genre)",
  "Books Read Count (Textbook Genre)",
  "Books Read Count (Romance Genre)",
  "Books Read Count (Literature Genre)",
  "Books Read Count (Science Fiction and Fantasy Genre)",
  "Books Read Count (Mystery/Thriller Genre)",
  "Books Read Count (Teens Genre)",
  "Books Read Count (Entertainment Genre)",
  "Books Read Count (LGBTQ Genre)",
  "Books Read Count (Unknown Genre)",
  "Books Read Count (Religion/Spirituality Genre)",
  "Books Read Count (Business/Investing Genre)",
  "Books Read Count (Family Genre)",
  "Books Read Count (Food/Wine Genre)",
  "Books Read Count (Other Genre)",
  "Books Read Count (History Genre)",
  "Books Read Count (Biology Genre)",
  "Books Read Count (Nonfiction Rollup Genre)",
  "Books Read Count (Fiction Rollup Genre)",
  "Books Read Count (Health Genre)",
  "Books Read Count (Ho